In [46]:
from pathlib import Path
import pandas as pd
import sqlite3

In [47]:
cnx = sqlite3.connect('offers.sqlite3')

In [48]:
df_parsed = pd.read_sql_query("SELECT * FROM parsed", cnx)
df_parsed = df_parsed[
    (df_parsed['serie'] != "Other") 
    & (df_parsed['generationCPU'] != "Other")]

target_set = set(df_parsed['listingID'])

df_parsed.head()


,listingID,serie,generationCPU,familyCPU,RAM,storageSize,screenSize,year
1,1073823445,Air,M3,Base,16,512,13,0
2,1073822559,Air,M4,Base,16,256,13,2025
5,1073821889,Pro,M2,Base,16,256,13,2022
6,1073821855,Pro,M1,Base,8,512,13,0
7,54365199,Pro,M3,Base,8,512,14,2023


In [49]:
df_offers = pd.read_sql_query("SELECT * FROM offers", cnx)

df_offers = df_offers.loc[:, ['listingID', 'price', 'date', 'lastSeen']]
df_offers = df_offers[df_offers['listingID'].isin(target_set)]

df_offers["price"] = (
    df_offers["price"]
    .str.replace(r"\D", "", regex=True)
    .astype(int)
)

df_offers["date"] = pd.to_datetime(df_offers["date"])
df_offers["lastSeen"] = pd.to_datetime(df_offers["lastSeen"])

df_offers.head()

,listingID,price,date,lastSeen
1,1073823445,520,2026-08-13 18:55:13+02:00,2026-08-17 17:30:28.591097+00:00
2,1073822559,850,2026-08-13 16:22:11+02:00,2026-08-13 22:51:27.880163+00:00
5,1073821889,590,2026-08-13 12:08:09+02:00,2026-08-17 17:30:30.157570+00:00
6,1073821855,450,2026-08-13 11:58:40+02:00,2026-08-17 17:30:30.157609+00:00
7,54365199,999,2026-08-13 10:19:48+02:00,2026-08-17 17:30:30.157658+00:00


In [50]:
df_prices = pd.read_sql_query("SELECT * FROM offer_price_history", cnx)

df_prices = df_prices[df_prices['listingID'].isin(target_set)]

df_prices["price"] = (
    df_prices["price"]
    .str.replace(r"\D", "", regex=True)
    .astype(int)
)
df_prices["observedAt"] = pd.to_datetime(df_prices["observedAt"])

df_prices.head()

,listingID,price,observedAt
1,1073823445,520,2026-08-13 22:51:27.880084+00:00
2,1073822559,850,2026-08-13 22:51:27.880163+00:00
5,1073821889,590,2026-08-13 22:51:27.880307+00:00
6,1073821855,450,2026-08-13 22:51:27.880335+00:00
7,54365199,999,2026-08-13 22:51:27.880369+00:00


In [51]:
df_market = df_parsed.merge(df_offers, on='listingID')

df_market.head()

,listingID,serie,generationCPU,familyCPU,RAM,storageSize,screenSize,year,price,date,lastSeen
0,1073823445,Air,M3,Base,16,512,13,0,520,2026-08-13 18:55:13+02:00,2026-08-17 17:30:28.591097+00:00
1,1073822559,Air,M4,Base,16,256,13,2025,850,2026-08-13 16:22:11+02:00,2026-08-13 22:51:27.880163+00:00
2,1073821889,Pro,M2,Base,16,256,13,2022,590,2026-08-13 12:08:09+02:00,2026-08-17 17:30:30.157570+00:00
3,1073821855,Pro,M1,Base,8,512,13,0,450,2026-08-13 11:58:40+02:00,2026-08-17 17:30:30.157609+00:00
4,54365199,Pro,M3,Base,8,512,14,2023,999,2026-08-13 10:19:48+02:00,2026-08-17 17:30:30.157658+00:00
